In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

secret = os.getenv("SECRET")
id = os.getenv("ID")


In [ ]:
from PIL import Image

# Load images
im = Image.open("C:/wallpapers/cover.jpg")   # target size
pippidon = Image.open("../assets/pippi.png") # smaller image

# Target size
target_size = im.size  # (900, 250)

# Create transparent background
new_im = Image.new("RGBA", target_size, (0, 0, 0, 0))

# Calculate center position
x = (target_size[0] - pippidon.width) // 2
y = (target_size[1] - pippidon.height) // 2

# Paste with transparency
new_im.paste(pippidon, (x, y), pippidon)

# Save result
new_im.save()


In [15]:
from osu import Client, GameModeStr

client = Client.from_credentials(id, secret, None)
res = client.search_beatmapsets(filters={"mode": GameModeStr.STANDARD, "query": "title=okurina mapper=itswinter"})
for _b in res.beatmapsets:
    print(_b.title)

Okurina -Inochi o Aoku Mimamori Nemurina-
Okurina -Inochi o Aoku Mimamori Nemurina-


In [2]:
from osu import Client, GameModeStr

client = Client.from_credentials(id, secret, None)
res = client.search_beatmapsets(filters={"m": GameModeStr.STANDARD})
beatmapsets = []
import time
from requests.exceptions import ConnectionError

for i in range(0, 21):
    while True:  # retry until success
        try:
            beatmapsets.extend(
                client.search_beatmapsets(
                    filters={"m": GameModeStr.STANDARD}, page=i
                ).beatmapsets
            )
            print(f"Finished page: {i}")
            break  # success, exit retry loop
        except ConnectionError:
            print(f"Connection error on page {i}, retrying in 5s...")
            time.sleep(5)


Finished page: 0
Finished page: 1
Finished page: 2
Finished page: 3
Finished page: 4
Finished page: 5
Finished page: 6
Finished page: 7
Finished page: 8
Finished page: 9
Finished page: 10
Finished page: 11
Finished page: 12
Finished page: 13
Finished page: 14
Finished page: 15
Finished page: 16
Finished page: 17
Finished page: 18
Finished page: 19
Finished page: 20


In [3]:
import os
import json
from datetime import datetime

existing_ids = []
for db_fpaths in [os.path.join("../db", fn) for fn in os.listdir("../db") if fn != "info.json"]:
    with open(db_fpaths, "r", encoding="utf-8") as db_fp:
        existing_ids.extend([item["id"] for item in json.load(db_fp)])
        db_fp.close()
print(f"Number of existing beatmaps: {len(existing_ids)}")

beatmap_ids = []
for item in beatmapsets:
    for b in item.beatmaps:
        if 4 <= b.difficulty_rating and not b.id in existing_ids:
            beatmap_ids.append((b.id, b.beatmapset_id, item.tags, datetime.fromisoformat(item.ranked_date.isoformat()).replace(tzinfo=None).isoformat() + "Z"))

print(f"Number of new (unique) beatmaps: {len(beatmap_ids)}")


Number of existing beatmaps: 27023
Number of new (unique) beatmaps: 39


In [ ]:
import requests

tmpv = 0
new_files = []
existing_files = os.listdir("./osu-files")
for b_id,beatmapset_id, tags, date_ranked in beatmap_ids:
    while True:
        try:
            new_fn = f"{b_id}_{beatmapset_id}.osu"
            new_fpath = f"osu-files/{b_id}_{beatmapset_id}.osu"
            if new_fn in existing_files:
                tmpv += 1
                print(f"File: {new_fn} already exists! {tmpv}/{len(beatmap_ids)}")
                break
            osu_data = requests.get(f"https://osu.ppy.sh/osu/{b_id}").content.decode("utf-8-sig").splitlines()
            with open(new_fpath, "w", encoding="utf-8") as fp:
                fp.writelines(l + "\n" for l in osu_data)
                fp.close()
                tmpv += 1
                print(f"{tmpv}/{len(beatmap_ids)}")
                new_files.append((f"osu-files/{b_id}_{beatmapset_id}.osu", b_id, beatmapset_id))
                break
        except ConnectionError:
            print("Connection error. Retrying after 60s...")
            time.sleep(60)


File: 5393108_2464185.osu already exists! 1/39
File: 5393111_2464185.osu already exists! 2/39
File: 5393108_2464185.osu already exists! 3/39
File: 5393111_2464185.osu already exists! 4/39
File: 5463157_2479183.osu already exists! 5/39
File: 5463234_2479183.osu already exists! 6/39
File: 5259023_2363271.osu already exists! 7/39
File: 5357995_2363271.osu already exists! 8/39
File: 5340381_2423043.osu already exists! 9/39
File: 5434153_2477885.osu already exists! 10/39
File: 5434155_2477885.osu already exists! 11/39
File: 5434156_2477885.osu already exists! 12/39
File: 5333037_2443907.osu already exists! 13/39
File: 5337334_2443907.osu already exists! 14/39
File: 4981089_2096231.osu already exists! 15/39
File: 5006454_2096231.osu already exists! 16/39
File: 5206545_2398400.osu already exists! 17/39
File: 2154013_537860.osu already exists! 18/39
File: 2154014_537860.osu already exists! 19/39
File: 913942_422096.osu already exists! 20/39
File: 5400165_2289643.osu already exists! 21/39
File:

In [5]:
to_executor = [(f"osu-files/{id}_{beatmapset_id}.osu", beatmapset_id, f"https://assets.ppy.sh/beatmaps/{beatmapset_id}/covers/cover.jpg", f"https://osu.ppy.sh/beatmapsets/{beatmapset_id}#osu/{id}", tags, date_ranked) for id, beatmapset_id, tags, date_ranked in beatmap_ids]
to_executor[0]

('osu-files/5393108_2464185.osu',
 2464185,
 'https://assets.ppy.sh/beatmaps/2464185/covers/cover.jpg',
 'https://osu.ppy.sh/beatmapsets/2464185#osu/5393108',
 'kaneshiro palace theme pop instrumental ザ・ロイヤル p5r royal sound track atlus p5 ペルソナ５ jrpg rpg shin megami tensei videogame video game ost soundtrack leominexd leomine harumiii harumi akareh kowari nyukai',
 '2026-01-31T20:24:51Z')

In [6]:
if __name__ == "__main__":
    import os
    from concurrent.futures import ProcessPoolExecutor, as_completed
    from osu_file_parser import create_stats_entry

    fpaths = [os.path.join("osu-files", fn) for fn in os.listdir("osu-files")]
    calculated = []
    
    with ProcessPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(create_stats_entry, f, beatmapset_id, bg_url, url, tags, date_ranked, True) for f, beatmapset_id, bg_url, url, tags, date_ranked in to_executor]
        for fut in as_completed(futures):
            stats = fut.result()
            if isinstance(stats, dict):
                calculated.append(fut.result())

    print(f"length calculated: {len(calculated)}")


length calculated: 6


In [ ]:
import json
from datetime import datetime
import os
import copy

def get_last_file():
    update_files = []
    for fn in os.listdir("../db"):
        if "update" in fn and fn.endswith(".json"):
            update_files.append(fn)
    update_files.sort(key=lambda fn: datetime.strptime(fn.split("update-")[1].replace(".json", ""), "%Y-%m-%d"))
    if len(update_files):
        return f"../db/{update_files[-1]}"
    return None

last_update_file = get_last_file()
if last_update_file:
    print(f"Last update file: {last_update_file}")
    with open(last_update_file, "r", encoding="utf-8") as fp:
        last_up_data = json.load(fp)
        all_up_data = copy.deepcopy(last_up_data)
        all_up_data.extend(calculated)
        all_up_data = set([tuple(item.items()) for item in all_up_data])
        all_up_data = [dict(item) for item in all_up_data]
        
        print(f"Last update data: {len(last_up_data)}\nNew calculated data: {len(calculated)}\nTotal of previous two lists: {len(all_up_data)}")
else:
    all_up_data = calculated
chunks = [all_up_data[i:i+5000] for i in range(0, len(all_up_data), 5000)]
for i,c in enumerate(chunks):
    now = datetime.now().strftime("%Y-%m-%d")
    with open(f"../db/[{i}][new]update-{now}.json", "w") as ufp:
        json.dump(c, ufp, indent=2)
        print(f"New update file at: {now}. Length: {len(c)}")
        ufp.close()



Last update file: ../db/[0]update-2026-01-31.json
Last update data: 2638
New calculated data: 6
Total of previous two lists: 2638
New update file at: 2026-01-31. Length: 2638


In [26]:
### MUST ALWAYS RUN ###
import os
import json
import datetime

now = datetime.datetime.now().strftime("%Y-%m-%d")
path_to_db = './db'

with open(f"../db/info-{now}.json", "w", encoding="utf-8") as fp:
    urls = []
    for fn in os.listdir(f"../db"):
        if "info" in fn:
            continue
        urls.append(f"{path_to_db}/{fn}")
    json.dump(urls, fp, indent=2)
urls


['./db/0-2026-01-31-mk88__-lazer.json',
 './db/1-2026-01-31-mk88__-lazer.json',
 './db/2-2026-01-31-mk88__-lazer.json',
 './db/3-2026-01-31-mk88__-lazer.json',
 './db/4-2026-01-31-mk88__-lazer.json',
 './db/[0]update-2026-01-31.json']

In [25]:
import json
import datetime

def is_iso_date(s):
    try:
        datetime.datetime.fromisoformat(s.replace("Z", "+00:00"))
        return True
    except ValueError:
        return False


with open("C:/programming-tools/projects/MarisaCodes/aimer/threading/lazer.json", "r", encoding="utf-8") as fp:
    data = json.load(fp)
    data = [item for item in data if item["star_rating"] >= 4 and is_iso_date(item["date_ranked"])]
    unique_data = set([tuple(item.items()) for item in data])
    unique_data = [dict(item) for item in unique_data]
    print(len(unique_data))


now = datetime.datetime.now().strftime("%Y-%m-%d")

chunks = [unique_data[i:i+5000] for i in range(0, len(unique_data), 5000)]
for i,c in enumerate(chunks):
    with open(f"../db/{i}-{now}-mk88__-lazer.json", "w", encoding="utf-8") as f:
        json.dump(c, f, indent=2)

24280


In [3]:
import os
import json
dp = "../db_2026_06_02"
fns = [f"./db_2026_06_02/{fn}" for fn in os.listdir(dp) if fn != "info.json"]

grp_size = 10
groups = [fns[i:i+grp_size] for i in range(0, len(fns), grp_size)]

with open(os.path.join(dp, "info.json"), "w", encoding="utf-8") as info_fp:
    json.dump(groups, info_fp, ensure_ascii=False, indent=2)
